# SupplyMind — GRPO + TRL + Unsloth Training

**One-click training of a Qwen-2.5-0.5B agent on the SupplyMind OpenEnv environment.**

Stack: **Unsloth** (4-bit NF4) → **TRL** (`GRPOTrainer`) → **OpenEnv** (HTTP client to live env).

Runtime: ~25 minutes on a free T4. Outputs:
- LoRA adapter at `./grpo_supplymind_qwen05b/final/`
- `reward_curve.png` (axes labeled)
- `loss_curve.png`
- `baseline_vs_trained.png` (same axes)
- `training_log.csv`

**Run order**: just press `Runtime → Run all`. Cells are sequential and idempotent.

## 1. Install dependencies

In [ ]:
%%capture
!pip install -q unsloth
!pip install -q --upgrade --no-deps "trl>=0.12.0" peft accelerate bitsandbytes
!pip install -q matplotlib datasets requests

## 2. Set the SupplyMind env URL

The training script needs a running OpenEnv server. Easiest options:
- **Live HF Space** (default): `https://shaurya-noodle-supplymind.hf.space`
- **Local Docker**: `http://localhost:8000` (run `docker-compose up`)

In [ ]:
import os
os.environ['SUPPLYMIND_ENV_URL'] = 'https://shaurya-noodle-supplymind.hf.space'

import requests
r = requests.get(f"{os.environ['SUPPLYMIND_ENV_URL']}/health", timeout=15)
print('env health:', r.status_code, r.json())

## 3. Load Qwen-2.5-0.5B with Unsloth (4-bit NF4)

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
MODEL_NAME = 'unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('model loaded:', sum(p.numel() for p in model.parameters() if p.requires_grad), 'trainable params')

## 4. Build the prompt dataset by querying the live env

In [ ]:
import json, re, uuid, requests
from datasets import Dataset

ENV_URL = os.environ['SUPPLYMIND_ENV_URL']
VALID_ACTIONS = {'do_nothing', 'issue_supplier_alert', 'activate_backup_supplier',
                 'reroute_shipment', 'increase_safety_stock', 'expedite_order', 'hedge_commodity'}

SYSTEM_PROMPT = '''You are a supply-chain risk manager. Output exactly ONE action as JSON on a single line.
Valid action types: do_nothing | issue_supplier_alert | activate_backup_supplier | reroute_shipment | increase_safety_stock | expedite_order | hedge_commodity
Act EARLY during warning phase. Stay within budget. Use FREE alerts for intel before spending.'''

def build_prompt(obs):
    o = obs.get('observation', obs)
    user = o.get('compact_summary') or o.get('situation_summary', '')
    return f'<|system|>\n{SYSTEM_PROMPT}\n<|user|>\n{user}\n<|assistant|>\n'

def collect_prompts(n=200):
    rows = []
    for task in ['easy_typhoon_response', 'medium_multi_front', 'hard_cascading_crisis']:
        per_task = max(1, n // 3)
        for ep in range(per_task // 5):
            sid = f'colab_{uuid.uuid4().hex[:8]}'
            r = requests.post(f'{ENV_URL}/reset',
                              json={'task_id': task, 'session_id': sid}, timeout=30).json()
            obs = r.get('observation', r)
            for _ in range(5):
                rows.append({'prompt': build_prompt({'observation': obs}), 'task_id': task})
                step = requests.post(f'{ENV_URL}/step',
                                     json={'session_id': sid, 'action': {'action_type': 'do_nothing'}},
                                     timeout=30).json()
                obs = step.get('observation', step)
                if step.get('done'):
                    break
                if len(rows) >= n:
                    return rows
    return rows[:n]

rows = collect_prompts(200)
print(f'collected {len(rows)} prompts')
train_dataset = Dataset.from_list(rows)

## 5. Reward function — verifier first, env-step-based

In [ ]:
def parse_action(text):
    if not text: return {'action_type': 'do_nothing'}, False
    m = re.search(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', text)
    if not m: return {'action_type': 'do_nothing'}, False
    try:
        a = json.loads(m.group(0))
        if not isinstance(a, dict) or 'action_type' not in a: return {'action_type': 'do_nothing'}, False
        if a['action_type'] not in VALID_ACTIONS: return {'action_type': 'do_nothing'}, False
        return a, True
    except: return {'action_type': 'do_nothing'}, False

def reward_fn(prompts, completions, **kwargs):
    rewards = []
    for prompt, completion in zip(prompts, completions):
        try:
            action, ok = parse_action(completion)
            sid = f'reward_{uuid.uuid4().hex[:6]}'
            requests.post(f'{ENV_URL}/reset', json={'task_id': 'easy_typhoon_response',
                                                     'session_id': sid}, timeout=20)
            r = requests.post(f'{ENV_URL}/step', json={'session_id': sid, 'action': action}, timeout=20).json()
            env_r = float(r.get('observation', {}).get('reward', 0.0))
            fmt = 0.05 if ok else -0.20
            free = 0.02 if action['action_type'] == 'issue_supplier_alert' else 0.0
            rewards.append(env_r + fmt + free)
        except Exception as e:
            rewards.append(-1.0)
    return rewards

## 6. GRPO training loop

In [ ]:
from trl import GRPOConfig, GRPOTrainer
import time

config = GRPOConfig(
    output_dir='./grpo_supplymind_qwen05b',
    num_generations=4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    max_steps=200,
    save_steps=50,
    logging_steps=5,
    bf16=True,
    report_to='none',
    max_prompt_length=1024,
    max_completion_length=128,
)
trainer = GRPOTrainer(
    model=model, tokenizer=tokenizer, args=config,
    train_dataset=train_dataset, reward_funcs=[reward_fn],
)
t0 = time.time()
trainer.train()
print(f'training done in {time.time() - t0:.1f}s')

## 7. Save adapter + plot reward & loss curves

In [ ]:
trainer.save_model('./grpo_supplymind_qwen05b/final')

import matplotlib.pyplot as plt
import csv, os as _os
_os.makedirs('./results', exist_ok=True)

log = trainer.state.log_history
steps = [e.get('step', 0) for e in log if 'step' in e]
rewards = [e.get('reward', e.get('rewards/mean', 0.0)) for e in log if 'step' in e]
losses = [e.get('loss', 0.0) for e in log if 'step' in e]

with open('./results/training_log.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['step', 'mean_reward', 'loss'])
    for s, r, l in zip(steps, rewards, losses): w.writerow([s, r, l])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(steps, rewards, 'g-', linewidth=2, label='GRPO-trained Qwen-2.5-0.5B')
ax.set_xlabel('Training step'); ax.set_ylabel('Mean reward (env.step()) per batch')
ax.set_title('SupplyMind GRPO Training — Reward Curve')
ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout(); plt.savefig('./results/reward_curve.png', dpi=120); plt.show()

if any(losses):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(steps, losses, 'r-', linewidth=2, label='GRPO loss')
    ax.set_xlabel('Training step'); ax.set_ylabel('GRPO loss')
    ax.set_title('SupplyMind GRPO Training — Loss Curve')
    ax.grid(True, alpha=0.3); ax.legend()
    plt.tight_layout(); plt.savefig('./results/loss_curve.png', dpi=120); plt.show()

print('saved: ./grpo_supplymind_qwen05b/final/')
print('saved: ./results/reward_curve.png')
print('saved: ./results/loss_curve.png')
print('saved: ./results/training_log.csv')

## 8. Baseline vs trained — same-axes comparison

Runs 25 episodes with the **untrained** Qwen-2.5-0.5B and 25 with the **GRPO-trained** version, plots both on the same axes.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import numpy as np

def run_episode(m, tok, task='easy_typhoon_response', max_steps=30):
    sid = f'eval_{uuid.uuid4().hex[:6]}'
    r = requests.post(f'{ENV_URL}/reset', json={'task_id': task, 'session_id': sid}, timeout=30).json()
    obs = r.get('observation', r); total = 0.0
    for _ in range(max_steps):
        prompt = build_prompt({'observation': obs})
        inputs = tok(prompt, return_tensors='pt', truncation=True, max_length=1024).to(m.device)
        with torch.no_grad():
            out = m.generate(**inputs, max_new_tokens=80, do_sample=True,
                             temperature=0.7, pad_token_id=tok.eos_token_id)
        comp = tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        action, _ = parse_action(comp)
        step = requests.post(f'{ENV_URL}/step', json={'session_id': sid, 'action': action}, timeout=20).json()
        obs = step.get('observation', step); total += float(obs.get('reward', 0.0))
        if step.get('done'): break
    return total

# Baseline (untrained)
FastLanguageModel.for_inference(model)
base_tok = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct')
base_model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct',
                                                   torch_dtype=torch.float16, device_map='auto')
print('Running baseline (untrained Qwen-2.5-0.5B) for 25 episodes...')
baseline_returns = [run_episode(base_model, base_tok) for _ in range(25)]

# Trained
print('Running GRPO-trained for 25 episodes...')
trained_returns = [run_episode(model, tokenizer) for _ in range(25)]

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(len(baseline_returns)), baseline_returns, 'k--', linewidth=2,
        label=f'Baseline Qwen-2.5-0.5B (n=25, mean={np.mean(baseline_returns):.3f})')
ax.plot(range(len(trained_returns)), trained_returns, 'g-', linewidth=2,
        label=f'GRPO-trained (n=25, mean={np.mean(trained_returns):.3f})')
ax.set_xlabel('Evaluation episode')
ax.set_ylabel('Episode return (sum of step rewards)')
ax.set_title('SupplyMind — Baseline vs GRPO-Trained Agent (same axes)')
ax.grid(True, alpha=0.3); ax.legend(loc='best')
plt.tight_layout(); plt.savefig('./results/baseline_vs_trained.png', dpi=120); plt.show()

print(f'\nbaseline mean: {np.mean(baseline_returns):.3f}')
print(f'trained  mean: {np.mean(trained_returns):.3f}')
print(f'lift: +{np.mean(trained_returns) - np.mean(baseline_returns):.3f}')

## 9. (Optional) Push adapter to Hub

In [ ]:
# Set HF_TOKEN in Colab Secrets to enable
# from huggingface_hub import notebook_login; notebook_login()
# model.push_to_hub('Shaurya-Noodle/supplymind-grpo-qwen05b')
# tokenizer.push_to_hub('Shaurya-Noodle/supplymind-grpo-qwen05b')
pass